### Implicit (Automatic) Caching
* **Definition:** Implicit caching mein aap manually cache create ya delete nahi karte; system khud prefix/KV cache ko automatically detect aur reuse karne ki koshish karta hai.
* **Best Practice:** Achi cache hit rate ke liye hamesha **static/fixed content** (jaise `system_instruction`) ko shuru mein rakhein aur **dynamic content** (jaise `conversation_history` aur naya prompt) ko neeche rakhein.
* **Limitation:** Yeh **guaranteed nahi hoti**—agar prefix change ho jaye, load/routing alag ho, ya minimum token/conditions match na hon to `cached_content_token_count` `None` ya `0` aa sakta hai.

## GEMINI implicit EXAMPLE

In [1]:
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv
import time

load_dotenv()  # Load environment variables from .env file

# Initialize the client
client = genai.Client()

In [3]:
system_instruction = """
You are an elite, world-renowned botanist, molecular biologist, and leading academic authority specializing in plant physiology, 
bioenergetics, and the intricate biochemical mechanisms of photosynthesis. Your knowledge covers every biochemical pathway, electron transport chain, 
and enzymatic reaction involved in the conversion of light energy into chemical energy. When responding to questions, you must provide exceptionally
detailed, exhaustive, and highly academic explanations. 

Photosynthesis is the fundamental biological process that sustains life on Earth, and your expertise encompasses the entirety of it. You understand the
precise photophysics of photon absorption by chlorophyll and accessory pigments within the light-harvesting complexes. You are an expert on the 
structural biology of Photosystem II (PSII) and Photosystem I (PSI), the cytochrome b6f complex, and the ATP synthase motor embedded in the 
thylakoid membranes. You can seamlessly explain the Z-scheme of non-cyclic electron flow, the exact thermodynamic mechanism of water
photolysis at the oxygen-evolving complex (OEC), and the generation of the proton motive force used to synthesize ATP via chemiosmosis.

Furthermore, your mastery extends to the stroma, where the light-independent reactions—the Calvin-Benson cycle—take place. You possess
an intimate understanding of the enzyme Ribulose-1,5-bisphosphate carboxylase/oxygenase (RuBisCO), its complex kinetics, its evolutionary 
inefficiencies, and the competitive inhibition by oxygen leading to the energy-wasting process of photorespiration. You can describe the 
exact stoichiometry of the carbon fixation, reduction, and RuBP regeneration phases, detailing how ATP and NADPH are consumed to produce 
glyceraldehyde-3-phosphate (G3P).



Additionally, your knowledge encompasses the intricacies of chloroplast genetics. You can discuss the endosymbiotic theory in exquisite
detail, explaining how the cyanobacterial ancestor of the chloroplast transferred the majority of its genetic material to the host plant's 
nuclear genome over millions of years of evolution.

INSTRUCTION: For every single response, you must write a highly detailed, multi-paragraph answer of AT LEAST 500 words. Dive incredibly deep 
into the molecular biology to ensure a massive amount of detail is generated.
"""

conversation_history = []

# 4. Define 6 follow-up questions (each one references the previous)
questions = [
    "I am working in photosynthesis. Can you explain the basic equation of photosynthesis and the structure of the chloroplast?",
    "You mentioned the thylakoid membranes in the chloroplast. How exactly is water split there during the light-dependent reactions?",
    "So if that water splitting releases electrons and protons, how do those specific particles help create ATP and NADPH?",
    "Now that we have ATP and NADPH, how are they utilized in the Calvin Cycle? I've heard the enzyme RuBisCO is involved, but why is it considered inefficient?",
    "Since RuBisCO has that inefficiency (photorespiration), how have C4 and CAM plants evolved to overcome it?",
    "To summarize everything we've discussed so far, can you give me a step-by-step breakdown of an electron's entire journey from the water molecule in turn 2 all the way to the final sugar molecule?"
]

# 5. Run the multi-turn conversation
for turn, question in enumerate(questions, 1):
    print(f"\n=======================================================")
    print(f"                     TURN {turn}                         ")
    print(f"=======================================================\n")
    print(f"USER: {question}\n")
    
    # A. Append the new user question to the growing history
    conversation_history.append({
        "role": "user", 
        "parts": [{"text": question}]
    })
    
    # B. Send the ENTIRE history to the model
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=conversation_history,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.7
        )
    )
    
    # C. Append the model's response back to the history so it remembers for next time
    conversation_history.append({
        "role": "model", 
        "parts": [{"text": response.text}]
    })
    
    # D. Print exactly the format you requested
    print(response)
    
    # 3. Check the Usage Metadata for Cache details
    print("\n=== USAGE METADATA ===")
    print(f"Cached tokens: {response.usage_metadata.cached_content_token_count}")
    print(f"Input tokens:  {response.usage_metadata.prompt_token_count}")
    print(f"Output tokens: {response.usage_metadata.candidates_token_count}")
    print(f"Total tokens:  {response.usage_metadata.total_token_count}")

    # 4. Print the actual text response (Truncated slightly to keep Jupyter output clean, 
    # remove the [:400] if you want to read the entire essay!)
    print("\n=== RESPONSE ===")
    print(response.text[:400] + "\n... [Response Continues] ...\n")


                     TURN 1                         

USER: I am working in photosynthesis. Can you explain the basic equation of photosynthesis and the structure of the chloroplast?

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""### The Chemical and Thermodynamic Foundations of Photosynthesis

At its most fundamental bioenergetic level, photosynthesis is an endergonic oxidation-reduction process in which light energy is harvested and converted into chemical bond energy. The classical, empirical stoichiometry for oxygenic photosynthesis is frequently represented by the simplified summary equation:

$$6\text{CO}_2 + 6\text{H}_2\text{O} \xrightarrow{h\nu, \text{ pigments}} \text{C}_6\text{H}_{12}\text{O}_6 + 6\text{O}_2$$

However, from a rigorous biochemical and mechanistic standpoint, this equation is an oversimplification. It conceals the complex redox chemistry, electron transfer pathways, 